# R2-A3-S13 · Avance de Proyecto III — Implementación y pruebas
## Predicción de probabilidad *proxy* de fraude bancario

**Autores:** Pedro Pascual Murcia Vargas · Jhon Eduard Tinjacá Cruz · Julián David Silva Guzmán  
**Programa:** Especialización en Inteligencia Artificial — Universidad de Cundinamarca  
**Asignatura:** Aplicaciones en Deep Learning · **Profesor:** Wilson Mauro Rojas Reales  

> **Advertencia de alcance.** La etiqueta del ejercicio se deriva de `mora > 0 OR atraso > 0`; por tanto, representa **riesgo/morosidad proxy**, no fraude confirmado. El cuaderno no debe utilizarse para acusar, bloquear o negar automáticamente servicios financieros.

Este Colab implementa un flujo reproducible: contrato de datos → particiones sin fuga → preprocesamiento ajustado solo con *train* → Random Forest baseline → ResMLP en TensorFlow/Keras → calibración → selección de umbral → evaluación cerrada → gráficos y artefactos.

**Datos:** seleccione `DATA_MODE='upload'` y cargue `data.csv`. Si no se carga el archivo, puede usar `DATA_MODE='synthetic'` para una prueba de ingeniería determinista. Las métricas sintéticas no son evidencia del desempeño bancario real.


## 0. Instrucciones rápidas
1. En Google Colab, seleccione **Entorno de ejecución → Ejecutar todas**.
2. Para evaluación real, cambie `DATA_MODE` a `"upload"` y cargue el CSV de Kaggle.
3. Descargue `artifacts_colab.zip` al terminar y suba el notebook, métricas y gráficos al repositorio.
4. No publique credenciales de Kaggle ni el dataset sin revisar su licencia.


## Resultados de referencia incluidos en el informe

La corrida local sobre el CSV adjunto (SHA-256 `eaa70a7d69f69d8e671f019d89d6e05386ea845811db085d812e1f4fa55d7b0a`) encontró 8.399 filas crudas, eliminó 2.325 duplicados exactos y trabajó con 6.074 registros. La prevalencia del proxy `mora > 0 OR atraso > 0` fue 0,790.

- **Random Forest:** PR-AUC 0,923; ROC-AUC 0,792; F1 0,893; recall 0,992; FPR 0,864; Brier 0,133.
- **MLP local 128-64-32:** PR-AUC 0,905; ROC-AUC 0,763; F1 0,882; recall 0,989; FPR 0,953; Brier 0,142.

Estos valores sirven como referencia reproducible del paquete. Al ejecutar este notebook, la **ResMLP Keras** vuelve a entrenarse y genera sus propios artefactos; sus resultados deben compararse bajo el mismo protocolo. La alta prevalencia y FPR confirman que el rediseño debe priorizar una etiqueta de fraude confirmada y una restricción explícita de falsas alertas.


In [ ]:
#@title 1. Configuración y dependencias
import os, re, io, json, random, hashlib, time, warnings, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, precision_recall_curve, roc_curve)
from sklearn.calibration import calibration_curve

SEED = 42
DATA_MODE = 'upload'  #@param ['upload', 'synthetic']
MIN_RECALL = 0.80
COST_FN, COST_FP = 5, 1
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
np.random.seed(SEED); random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEED)
    print('TensorFlow:', tf.__version__)
except Exception as e:
    raise RuntimeError('Este notebook requiere TensorFlow/Keras. En Google Colab viene preinstalado.') from e
print('Semilla:', SEED)


## 2. Carga o generación de datos
El modo `upload` solicita `data.csv` en Colab. El modo `synthetic` conserva las 13 columnas fuente y genera 8.399 registros para comprobar que todo el código se ejecuta. No se mezclan resultados sintéticos con conclusiones sobre clientes reales.


In [ ]:
def normalize_name(s):
    import unicodedata
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+','_',s).strip('_')

def make_synthetic(n=8399, seed=SEED):
    rng=np.random.default_rng(seed)
    edad=np.clip(rng.normal(39,11,n),18,75).round()
    dias_lab=np.maximum(0,rng.gamma(2.2,850,n)); exp_sf=np.maximum(0,rng.gamma(2,34,n))
    ingreso=np.maximum(550,rng.lognormal(np.log(2500),.65,n))
    linea=np.maximum(0,ingreso*rng.lognormal(np.log(2),.55,n)); deuda=linea*rng.beta(2,2.7,n)
    score=np.clip(rng.normal(590,105,n),200,900)
    nivel_ahorro=rng.integers(0,13,n)
    vivienda=rng.choice(['PROPIA','FAMILIAR','ALQUILADA'],n,p=[.36,.34,.30])
    nivel=rng.choice(['SIN EDUCACION','SECUNDARIA','TECNICA','UNIVERSITARIA'],n,p=[.05,.42,.24,.29])
    zona=rng.choice(['LIMA','NORTE','CENTRO','SUR','ORIENTE'],n,p=[.42,.18,.14,.17,.09])
    sbs=rng.choice(['0','1','2','3','4'],n,p=[.57,.20,.12,.07,.04])
    util=deuda/(linea+1); dti=deuda/(ingreso+1)
    logit=(-2.55+2.4*util+.30*dti+.010*(590-score)+.52*np.isin(sbs,['2','3','4'])
           +.48*(vivienda=='ALQUILADA')+.22*(nivel=='SIN EDUCACION')+.15*(exp_sf<12)
           +rng.normal(0,.62,n))
    y=rng.binomial(1,1/(1+np.exp(-logit)))
    df=pd.DataFrame({'mora':np.where(y,rng.binomial(1,.70,n),0),
        'atraso':np.where(y,rng.integers(1,241,n),0),'vivienda':vivienda,'edad':edad,
        'dias_lab':dias_lab.round(),'exp_sf':exp_sf.round(1),'nivel_ahorro':nivel_ahorro,'ingreso':ingreso.round(2),
        'linea_sf':linea.round(2),'deuda_sf':deuda.round(2),'score':score.round(),
        'zona':zona,'clasif_sbs':sbs,'nivel_educ':nivel})
    for c,p in [('exp_sf',.035),('linea_sf',.028),('deuda_sf',.025),('vivienda',.012),('nivel_educ',.010)]:
        df.loc[rng.choice(n,int(n*p),replace=False),c]=np.nan
    return df

def load_data(mode):
    if mode=='synthetic':
        df=make_synthetic(); raw=df.to_csv(index=False).encode(); return df, 'synthetic_engineering_demo', hashlib.sha256(raw).hexdigest()
    try:
        from google.colab import files
        uploaded=files.upload(); name=next(iter(uploaded)); raw=uploaded[name]
        return pd.read_csv(io.BytesIO(raw)), 'real_uploaded', hashlib.sha256(raw).hexdigest()
    except Exception as e:
        raise RuntimeError("No se cargó data.csv. Cambie DATA_MODE a 'synthetic' para la prueba técnica.") from e

df, DATA_SOURCE, DATA_HASH = load_data(DATA_MODE)
df.columns=[normalize_name(c) for c in df.columns]
RAW_ROWS=len(df); DUPLICATES=int(df.duplicated().sum())
df=df.drop_duplicates().reset_index(drop=True)
print('Fuente:',DATA_SOURCE,'| Filas crudas:',RAW_ROWS,'| Duplicados eliminados:',DUPLICATES,'| Dimensión analítica:',df.shape)
display(df.head())


## 3. Contrato, etiqueta y controles de fuga
Las columnas `mora` y `atraso` se usan una sola vez para construir el objetivo y después se eliminan de los predictores. También se verifica dominio, duplicados, faltantes y presencia de las variables mínimas.


In [ ]:
REQUIRED=['mora','atraso','vivienda','edad','dias_lab','exp_sf','nivel_ahorro','ingreso','linea_sf','deuda_sf','score','zona','clasif_sbs','nivel_educ']
missing=sorted(set(REQUIRED)-set(df.columns))
assert not missing, f'Faltan columnas obligatorias: {missing}'
assert (pd.to_numeric(df['mora'],errors='coerce').fillna(0)>=0).all()
assert (pd.to_numeric(df['atraso'],errors='coerce').fillna(0)>=0).all()
y=((pd.to_numeric(df['mora'],errors='coerce').fillna(0)>0)|
   (pd.to_numeric(df['atraso'],errors='coerce').fillna(0)>0)).astype('int8')
X=df.drop(columns=['mora','atraso']).copy()
assert not {'mora','atraso'}.intersection(X.columns)
quality=pd.DataFrame({'tipo':df.dtypes.astype(str),'nulos':df.isna().sum(),
                      'pct_nulos':(100*df.isna().mean()).round(2),'unicos':df.nunique(dropna=True)})
print('Prevalencia proxy:',round(y.mean(),4),'| duplicados:',int(df.duplicated().sum()))
display(quality)


## 4. Ingeniería y particiones
Después de eliminar duplicados exactos, se agregan razones financieras y se separan cuatro subconjuntos: 70% entrenamiento, 10% calibración, 5% selección de umbral y 15% prueba. El conjunto de prueba no participa en ajuste, calibración ni umbral.


In [ ]:
def add_features(z):
    z=z.copy()
    z['ratio_deuda_ingreso']=z['deuda_sf']/(z['ingreso'].abs()+1)
    z['utilizacion_credito']=z['deuda_sf']/(z['linea_sf'].abs()+1)
    z['antiguedad_laboral_anios']=z['dias_lab']/365.25
    z['experiencia_financiera_anios']=z['exp_sf']/12
    return z.replace([np.inf,-np.inf],np.nan)

def split_sets(X,y):
    Xtr,Xtmp,ytr,ytmp=train_test_split(X,y,test_size=.30,stratify=y,random_state=SEED)
    Xcal,Xtmp2,ycal,ytmp2=train_test_split(Xtmp,ytmp,test_size=2/3,stratify=ytmp,random_state=SEED)
    Xthr,Xte,ythr,yte=train_test_split(Xtmp2,ytmp2,test_size=.75,stratify=ytmp2,random_state=SEED)
    return Xtr,Xcal,Xthr,Xte,ytr,ycal,ythr,yte

X=add_features(X)
X_train,X_cal,X_thr,X_test,y_train,y_cal,y_thr,y_test=split_sets(X,y)
print({k:len(v) for k,v in {'train':y_train,'calibration':y_cal,'threshold':y_thr,'test':y_test}.items()})


## 5. Preprocesamiento sin fuga
Numéricas: mediana, indicadores de faltantes y estandarización. Categóricas: moda y one-hot con categorías desconocidas toleradas. El preprocesador se ajusta exclusivamente con entrenamiento.


In [ ]:
num_cols=X.select_dtypes(include=np.number).columns.tolist()
cat_cols=[c for c in X.columns if c not in num_cols]
preprocessor=ColumnTransformer([
 ('num',Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),
                  ('scaler',StandardScaler())]),num_cols),
 ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                  ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat_cols)
],sparse_threshold=0)
Xtr=preprocessor.fit_transform(X_train).astype('float32')
Xca=preprocessor.transform(X_cal).astype('float32')
Xth=preprocessor.transform(X_thr).astype('float32')
Xte=preprocessor.transform(X_test).astype('float32')
print('Entrada original:',X_train.shape,'→ tensor:',Xtr.shape)


## 6. Baseline: Random Forest
El baseline conserva comparabilidad con las entregas anteriores, usa pesos balanceados y probabilidades para aplicar exactamente el mismo protocolo de calibración y umbral.


In [ ]:
rf=RandomForestClassifier(n_estimators=350,min_samples_leaf=4,class_weight='balanced_subsample',
                          n_jobs=-1,random_state=SEED)
rf.fit(Xtr,y_train)
print('Random Forest entrenado.')


## 7. Modelo profundo: ResMLP tabular
La red usa una proyección inicial de 128 unidades, Batch Normalization, GELU, dropout, dos bloques residuales, compresión a 64 unidades y salida sigmoide. AdamW y *early stopping* controlan optimización y sobreajuste. Los pesos de clase se calculan solo con entrenamiento.


In [ ]:
def residual_block(x, units=128, dropout=.20):
    shortcut=x
    x=tf.keras.layers.Dense(units)(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.Activation('gelu')(x)
    x=tf.keras.layers.Dropout(dropout)(x)
    x=tf.keras.layers.Dense(units)(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.Add()([shortcut,x])
    return tf.keras.layers.Activation('gelu')(x)

def build_resmlp(n_features):
    inp=tf.keras.Input(shape=(n_features,),name='features')
    x=tf.keras.layers.Dense(128)(inp)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.Activation('gelu')(x)
    x=tf.keras.layers.Dropout(.20)(x)
    x=residual_block(x); x=residual_block(x)
    x=tf.keras.layers.Dense(64,activation='gelu')(x)
    x=tf.keras.layers.Dropout(.20)(x)
    out=tf.keras.layers.Dense(1,activation='sigmoid',name='probabilidad_proxy')(x)
    model=tf.keras.Model(inp,out,name='ResMLP_fraude_proxy')
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3,weight_decay=1e-4),
        loss='binary_crossentropy',metrics=[tf.keras.metrics.AUC(curve='PR',name='pr_auc'),
        tf.keras.metrics.AUC(curve='ROC',name='roc_auc'),tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')])
    return model

neg,pos=np.bincount(np.asarray(y_train)); class_weight={0:1.0,1:float(neg/max(pos,1))}
resmlp=build_resmlp(Xtr.shape[1]); resmlp.summary()
history=resmlp.fit(Xtr,y_train,validation_split=.15,epochs=150,batch_size=256,
    class_weight=class_weight,verbose=0,callbacks=[
      tf.keras.callbacks.EarlyStopping(monitor='val_pr_auc',mode='max',patience=12,
                                       restore_best_weights=True,min_delta=1e-4),
      tf.keras.callbacks.ReduceLROnPlateau(monitor='val_pr_auc',mode='max',patience=5,factor=.5,min_lr=1e-5)])
print('Épocas ejecutadas:',len(history.history['loss']))


## 8. Calibración, umbral y métricas
La calibración isotónica se ajusta en su subconjunto independiente. El umbral se selecciona en otro subconjunto buscando el mejor F1 entre los puntos que cumplen `recall ≥ 0,80`. Después se evalúa una sola vez en prueba. Además de accuracy se reportan PR-AUC, ROC-AUC, precisión, recall, F1, balanced accuracy, Brier, FPR y costo esperado.


In [ ]:
def predict_prob(model,X,name):
    return model.predict_proba(X)[:,1] if name=='Random Forest' else model.predict(X,verbose=0).ravel()
def choose_threshold(y,p,min_recall=MIN_RECALL):
    rows=[]
    for t in np.linspace(.01,.99,199):
        pred=p>=t; rows.append((t,recall_score(y,pred,zero_division=0),f1_score(y,pred,zero_division=0)))
    feasible=[r for r in rows if r[1]>=min_recall]
    return max(feasible,key=lambda r:r[2])[0] if feasible else max(rows,key=lambda r:r[2])[0]
def evaluate(y,p,t):
    pred=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),
      'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),
      'f1':f1_score(y,pred,zero_division=0),'roc_auc':roc_auc_score(y,p),
      'pr_auc':average_precision_score(y,p),'brier':brier_score_loss(y,p),'threshold':float(t),
      'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp),'fpr':fp/max(fp+tn,1),
      'expected_cost':int(COST_FN*fn+COST_FP*fp)}

models={'Random Forest':rf,'ResMLP':resmlp}; metrics={}; test_probs={}
for name,model in models.items():
    pcal=predict_prob(model,Xca,name); pthr_raw=predict_prob(model,Xth,name); ptest_raw=predict_prob(model,Xte,name)
    calibrator=IsotonicRegression(out_of_bounds='clip').fit(pcal,y_cal)
    pthr=calibrator.transform(pthr_raw); ptest=calibrator.transform(ptest_raw)
    threshold=choose_threshold(y_thr,pthr); metrics[name]=evaluate(y_test,ptest,threshold); test_probs[name]=ptest
results=pd.DataFrame(metrics).T
print('Fuente:',DATA_SOURCE); display(results.round(4))


## 9. Visualizaciones y diagnóstico
Las curvas de precisión-recall y ROC evalúan ranking; la matriz de confusión muestra errores al umbral congelado; la curva de calibración contrasta probabilidad y frecuencia observada; las curvas de aprendizaje permiten detectar sobreajuste.


In [ ]:
sns.set_theme(style='whitegrid')
fig,axes=plt.subplots(2,2,figsize=(12,9))
for name,p in test_probs.items():
    pr,re,_=precision_recall_curve(y_test,p); axes[0,0].plot(re,pr,lw=2,label=f"{name} AP={metrics[name]['pr_auc']:.3f}")
    fpr,tpr,_=roc_curve(y_test,p); axes[0,1].plot(fpr,tpr,lw=2,label=f"{name} AUC={metrics[name]['roc_auc']:.3f}")
axes[0,0].axhline(y_test.mean(),ls='--',c='grey'); axes[0,0].set(xlabel='Recall',ylabel='Precisión',title='Curva precisión-recall'); axes[0,0].legend()
axes[0,1].plot([0,1],[0,1],'--',c='grey'); axes[0,1].set(xlabel='FPR',ylabel='TPR',title='Curva ROC'); axes[0,1].legend()
for ax,(name,p) in zip(axes[1],test_probs.items()):
    cm=confusion_matrix(y_test,p>=metrics[name]['threshold'],labels=[0,1]); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax)
    ax.set(xlabel='Predicción',ylabel='Real',title=f"{name} — umbral {metrics[name]['threshold']:.2f}")
fig.tight_layout(); fig.savefig(ARTIFACTS/'evaluation.png',dpi=160,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(7,5))
for name,p in test_probs.items():
    frac,mean=calibration_curve(y_test,p,n_bins=10,strategy='quantile'); ax.plot(mean,frac,marker='o',label=f"{name} Brier={metrics[name]['brier']:.3f}")
ax.plot([0,1],[0,1],'--',c='black'); ax.set(xlabel='Probabilidad media',ylabel='Frecuencia observada',title='Calibración'); ax.legend();
fig.tight_layout(); fig.savefig(ARTIFACTS/'calibration.png',dpi=160,bbox_inches='tight'); plt.show()

pd.DataFrame(history.history)[['loss','val_loss','pr_auc','val_pr_auc']].plot(subplots=True,figsize=(9,6),grid=True,title=['Pérdida','PR-AUC'])
plt.tight_layout(); plt.savefig(ARTIFACTS/'learning_curves.png',dpi=160,bbox_inches='tight'); plt.show()


## 10. Pruebas automáticas de humo
Estas aserciones comprueban ausencia de fuga directa, dimensiones, rango de probabilidades, finitud y persistencia de artefactos. En GitHub deben complementarse con `pytest` y CI.


In [ ]:
assert not {'mora','atraso'}.intersection(X.columns)
assert Xtr.shape[0]==len(y_train) and Xte.shape[0]==len(y_test)
for name,p in test_probs.items():
    assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all(), name
assert set(results.index)=={'Random Forest','ResMLP'}
results.to_csv(ARTIFACTS/'metrics.csv',index_label='model')
run_metadata={'data_source':DATA_SOURCE,'seed':SEED,'data_sha256':DATA_HASH,'raw_rows':RAW_ROWS,'duplicate_rows_removed':DUPLICATES,'rows':len(df),'positive_rate':float(y.mean()),
              'splits':{'train':len(y_train),'calibration':len(y_cal),'threshold':len(y_thr),'test':len(y_test)},
              'metrics':metrics,'warning':'Proxy mora/atraso; no fraude confirmado.'}
(ARTIFACTS/'metrics.json').write_text(json.dumps(run_metadata,ensure_ascii=False,indent=2))
resmlp.save(ARTIFACTS/'resmlp.keras')
print('Pruebas de humo: OK')


## 11. Interpretación y propuesta de rediseño
Use la tabla anterior para responder:
- ¿Cuál modelo tiene mayor PR-AUC y menor Brier?
- ¿El recall ≥ 0,80 se logra con un FPR/costo aceptable?
- ¿Las curvas de entrenamiento divergen?
- ¿La diferencia es estable por semilla, subgrupo y periodo?

**Rediseño propuesto si la red no supera al baseline o genera demasiadas alertas:**
1. Reemplazar one-hot por embeddings categóricos y conservar máscaras de faltantes.
2. Ajustar profundidad, dropout, learning rate, weight decay y tamaño de lote con Optuna; comparar BCE ponderada y focal loss.
3. Optimizar umbral con restricción explícita de FPR/capacidad, no solo recall/F1.
4. Comparar CatBoost/XGBoost y un EBM interpretable bajo las mismas particiones.
5. Usar validación temporal cuando existan fechas; repetir semillas y bootstrap.
6. Evaluar `clasif_sbs`, zona y educación mediante ablación, circularidad y equidad.
7. Recalibrar en un conjunto independiente y monitorizar Brier/ECE, deriva y carga de analistas.
8. Mantener revisión humana; no convertir el proxy en acusación de fraude.


In [ ]:
#@title 12. Empaquetar resultados
with zipfile.ZipFile('artifacts_colab.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in ARTIFACTS.rglob('*'):
        if p.is_file(): z.write(p,p.as_posix())
print('Creado: artifacts_colab.zip')
try:
    from google.colab import files
    files.download('artifacts_colab.zip')
except Exception:
    pass


## 13. Lista de verificación para GitHub
- `notebooks/Avance_Proyecto_III_Fraude_Bancario.ipynb`
- `src/` con funciones reutilizables
- `tests/` y `.github/workflows/ci.yml`
- `results/metrics.json`, `metrics.csv` y gráficos
- `docs/Informe_Avance_Proyecto_III_Fraude_Bancario.docx`
- `README.md`, `requirements.txt`, `.gitignore` y licencia
- URL final del repositorio en el informe y en la plataforma académica

### Referencias base
Calderón Baldeón, L. H. (2021). *BankDefaultAnalysis* [Dataset]. Kaggle. https://doi.org/10.34740/KAGGLE/DSV/2528417  
Kang, J., & Kim, K. (2026). Deployment-oriented interpretable fraud detection via hybrid explainable boosting machines with concept-raw fusion. *Applied Sciences, 16*(12), 5809.  
Documentación de TensorFlow/Keras y scikit-learn consultada para la implementación.
